In [ ]:
#base environment for python scripts

In [ ]:
import sys
import os
import argparse
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [ ]:
#global variables

In [ ]:
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

In [ ]:
#load peaks and hits

In [ ]:
X_peaks_coords = pd.read_csv("~/Treeshrew/FileSave/TreeShrew_peak_1000bp_OBO/modified_peak/C3.1_W1000bp_Peak.modified.bed", sep='\t', header=None, names = NARROWPEAK_SCHEMA)
X_peaks_coords.head()

In [ ]:
X_peaks_coords.shape

In [ ]:
X_peaks_coords["1"]= X_peaks_coords["chr"] + ":" + (X_peaks_coords["start"]+1).astype(str) + "-" + X_peaks_coords["end"].astype(str)
X_peaks_coords.head()

In [ ]:
X_peaks_coords["2"]=X_peaks_coords.index
X_peaks_coords.head()

In [ ]:
marker_W1000_peak_bed=pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/data/region_for_predict/C3.1_peak/C3.1_marker_peaks.modified.bed",sep='\t', header=None, names = NARROWPEAK_SCHEMA)
marker_W1000_peak_bed

In [ ]:
#marker_W1000_peak_bed["1"]= marker_W1000_peak_bed["chr"] + ":" + (marker_W1000_peak_bed["start"]+1).astype(str) + "-" + marker_W1000_peak_bed["end"].astype(str)
marker_W1000_peak_bed["1"]= marker_W1000_peak_bed["chr"] + ":" + (marker_W1000_peak_bed["start"]).astype(str) + "-" + marker_W1000_peak_bed["end"].astype(str)
marker_W1000_peak_bed.head()

In [ ]:
marker_W1000_peak_bed.shape

In [ ]:
keep_W1000_peak=marker_W1000_peak_bed["1"].unique().tolist()
len(keep_W1000_peak)

In [ ]:
X_peaks_coords_filter = X_peaks_coords[X_peaks_coords["1"].isin(keep_W1000_peak)]
X_peaks_coords_filter.shape

In [ ]:
X_peaks_coords_filter.head()

In [ ]:
hits = pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.1/finemo/counts/hit_call/hits.tsv", sep="\t")
hits.head()

In [ ]:
hits = hits[hits['peak_id'].isin(X_peaks_coords.index)]
max(hits['peak_id'])

In [ ]:
#hits['hit_coefficient_global'].describe()

In [ ]:
#hits=hits[hits["hit_coefficient_global"]>hits['hit_coefficient_global'].quantile(q=0.25)]
#hits

In [ ]:
hits_filter = hits[hits['peak_id'].isin(X_peaks_coords_filter['2'])]
hits_filter

In [ ]:
#hits_filter=hits_filter[hits_filter["hit_coefficient"]>10]
#hits_filter

In [ ]:
motif_matrix = pd.crosstab(hits_filter['peak_id'], hits_filter['motif_name'])
motif_matrix.head()

In [ ]:
motif_matrix.sum(axis=1).describe()

In [ ]:
max(motif_matrix.sum(axis=1))

In [ ]:
column_sums = motif_matrix.sum()
column_sums
# 按总和降序排序
sorted_columns = column_sums.sort_values(ascending=False)
sorted_columns
top_6_columns = sorted_columns.head(6).index.tolist()
top_6_columns

In [ ]:
motif_matrix.to_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.1/finemo/marker_peak/counts/TS_C3.1_markerW1000peaks_motif_presence_matrix.csv")

In [ ]:
#Create Binary Matrix for Co-occurrence (Present or Not)二值矩阵
motif_binary = (motif_matrix > 0).astype(int)

In [ ]:
# Calculate Co-occurrence Matrix共现矩阵
# (Transpose) dot (Matrix) -> (Motifs x Peaks) dot (Peaks x Motifs) -> (Motifs x Motifs)
cooccurrence_matrix = motif_binary.T.dot(motif_binary)

In [ ]:
cooccurrence_matrix.to_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.1/finemo/marker_peak/counts/TS_C3.1_markerW1000peaks_motif_cooccurrence_matrix.csv")

In [ ]:
custom_colors = ["#FDCDA1", "#FF99C9", "#9970A1", "#9999F9",  # 1-4
                "#F1A039", "#59B9D2", "#2A6364", "#F0DFF9",  # 5-8
                "#EB5A5F", "#76EEB0", "#56BC74"]    

In [ ]:
import matplotlib.pyplot as plt
# 1. 定义感兴趣的 Motif 列表
#C3.1条件------
interested_motifs = ["pos_patterns.pattern_2","pos_patterns.pattern_3","pos_patterns.pattern_5"] 
required_motif = "pos_patterns.pattern_3"
optional_motifs = ["pos_patterns.pattern_2", "pos_patterns.pattern_5"]
#C3.2条件------
#interested_motifs = ["pos_patterns.pattern_2","pos_patterns.pattern_4","pos_patterns.pattern_5"] 
#required_motif = "pos_patterns.pattern_5"
#optional_motifs = ["pos_patterns.pattern_2", "pos_patterns.pattern_4"]

# 2. 筛选数据
motif_matrix=motif_matrix[motif_matrix.sum(axis=1)>5] #要求motif总数大于5
filtered_matrix = motif_matrix[(motif_matrix[required_motif] > 0) & (motif_matrix[optional_motifs].sum(axis=1) > 0)]#出现required motif，并且至少出现一个optional motif
# 2.1 只取出感兴趣的列
subset_matrix = filtered_matrix[interested_motifs]


# 2.2 找出包含这些 Motif 总数最多的前 100 个 Peak (按行求和排序)
top_100_peaks = subset_matrix.loc[subset_matrix.sum(axis=1).sort_values(ascending=False).head(100).index]

# 显示数据概览
print("Top 100 Peaks with interested motifs:")
print(top_100_peaks)

# 3. 画堆叠柱状图 (Stacked Bar Chart)
#设置图表大小
plt.figure(figsize=(12, 6))
# 直接使用 pandas 的绘图功能，kind='bar', stacked=True
top_100_peaks.plot(kind='bar', stacked=True, figsize=(12, 6), color=custom_colors[:len(top_100_peaks.columns)])

plt.title('Occurrence of Interested Motifs in Top 100 Peaks')
plt.xlabel('Peak ID')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right') # 旋转X轴标签，防止重叠
plt.legend(title='Motif Name', bbox_to_anchor=(1.05, 1), loc='upper left') # 图例放外侧
plt.tight_layout()

plt.show()

In [ ]:
X_peaks_coords_filter[X_peaks_coords_filter["2"].isin(top_100_peaks.index)]

In [ ]:
X_peaks_coords_filter[X_peaks_coords_filter["2"].isin(top_100_peaks.index)].to_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/data/region_for_predict/C3.1_peak/C3.1_marker_peaks_filter_with_motif.bed", sep="\t", index=False, header=False)